# Bounded P3 refinement — crop1.5 + SF

Primary base: notebook13 **P3_SF_crop15**, preserved as the read-only reference. Mandatory runs are **P3_SF_W35_crop15** and **P3_SF_W50_crop15**. These reuse historical notebook06 weight ideas under newer crop1.5+SF geometry.

1. Edit resource/image/model paths; attach the existing frozen Train10K/Val3K and pooled LCC bundle.
2. Run setup, then the **source geometry preflight** cell. It uses Train10K image headers and the exact epoch2 training jitter/crop helper. Window translation is distinguished from true effective-scale reduction.
3. Inspect the preflight summary. DSF is appended to enabled runs only after the frozen source-only non-degeneracy gate passes; no replacement third experiment is added.
4. Run the report/export cell to obtain a preflight ZIP before training, then the final runner to train the predefined selected runs. `RUN_FILTER=None` runs both mandatory variants plus DSF only if enabled. Explicit subsets support separate Kaggle sessions.

SF centers/sigma, gain, model/init, augmentation, SGD/batch256/max20/milestones[6,14]/early-stop8/patience4 and one clean warm-up remain fixed. Active loss reads actual run coefficients. No R7, severity, harmful gating, full CelebA Test, detector or target fitting.

Optional historical summary paths are read only in reporting. Supplied C/SF pooled AUC values are labeled as user-provided AUC-only references until full artifacts are attached. This bounded one-seed micro round does not search until a target is reached or declare a final winner.


In [ ]:
# Edit paths here. Both notebooks use the same frozen 00d resource bundle.
RESOURCE_ROOT='/kaggle/input/datasets/maithanhphuong/full-resources-v3/minifasnet_binary_crossdomain_resources_v2_lcc_pooled'
CELEBA_ROOT='/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof'
LCC_ROOT='/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD'
MINIFASNET_SOURCE_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py'
MINIFASNET_CHECKPOINT_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth'
OUTPUT_ROOT='/kaggle/working/micro_P3_refine_crop15_v1'
LOG_EVERY_BATCHES=10
RESUME=True
NUM_WORKERS=4
AMP=True
DATA_SEED=43
TRAIN_SEED=100
BATCH_SIZE=256
MAX_EPOCHS=20
MILESTONES=[6,14]
GAMMA=.2
EARLIEST_EARLY_STOP_EPOCH=8
EARLY_STOP_PATIENCE=4
WARMUP_EPOCHS=1
CSMR_PRESERVE_DC=True
SELECTION_CHUNK_SIZE=32
CROP_SCALE=1.5
TRAIN_CROP_FACTOR=1.5
EVAL_CROP_FACTOR=1.5
INPUT_SIZE=80
METRIC_TIE_EPS=1e-12



RUN_FILTER=None  # Mandatory runs plus DSF ONLY if Train10K preflight enables it.
HISTORICAL_C_CROP15_SUMMARY=None
HISTORICAL_P3_SF_CROP15_SUMMARY=None
HISTORICAL_CROP27_WEIGHT_SUMMARY=None  # Optional notebook06 CSV/JSON, contextual only.
# These AUC-only reference values were supplied by the user; attach measured summaries for other metrics.
PROVIDED_REFERENCE_AUC={'C_crop15':.830642,'P3_SF_crop15':.837525}
DESCRIPTIVE_SCREENING_TARGET=.840642  # No new run/search is added to reach this value.
REFERENCE_CROP=2.7
REFERENCE_SIGMA=.10
ORIGINAL_CENTERS=[.15,.45,.75]
SCALE_RATIO=1.5/REFERENCE_CROP
SCALED_CENTERS=[c*SCALE_RATIO for c in ORIGINAL_CENTERS]
SCALED_SIGMA=REFERENCE_SIGMA*SCALE_RATIO
PREFLIGHT_EPOCH=2  # Exact training jitter at the first spectral-active epoch.
DSF_SCALE_TOLERANCE=.01
DSF_MIN_VARIATION_FRACTION=.01
DSF_MIN_SCALE_STD=.005
RUNS={
 'P3_SF_W35_crop15':{'family':'P3','method':'WORST','crop_factor':1.5,'centers':SCALED_CENTERS,'sigma':SCALED_SIGMA,'geometry_mode':'STATIC','clean_ce_weight':.65,'spectral_ce_weight':.35},
 'P3_SF_W50_crop15':{'family':'P3','method':'WORST','crop_factor':1.5,'centers':SCALED_CENTERS,'sigma':SCALED_SIGMA,'geometry_mode':'STATIC','clean_ce_weight':.50,'spectral_ce_weight':.50},
}
DSF_SPEC={'family':'P3','method':'WORST','crop_factor':1.5,'centers':ORIGINAL_CENTERS,'sigma':REFERENCE_SIGMA,'geometry_mode':'DYNAMIC','clean_ce_weight':.75,'spectral_ce_weight':.25}
REPORT_STEM='p3_refinement'
def validate_requested_filter(run_filter):
    allowed={'P3_SF_W35_crop15','P3_SF_W50_crop15','P3_DSF_crop15'}
    if run_filter is not None and (not isinstance(run_filter,(list,tuple)) or not run_filter or len(set(run_filter))!=len(run_filter) or set(run_filter)-allowed):
        raise ValueError('RUN_FILTER must be None or a non-empty unique subset of the two mandatory IDs and conditional P3_DSF_crop15')
def select_runs(run_filter):
    validate_requested_filter(run_filter)
    if run_filter is not None and 'P3_DSF_crop15' in run_filter and 'P3_DSF_crop15' not in RUNS:
        raise ValueError('P3_DSF_crop15 not enabled: source preflight found geometry effectively degenerate under current crop helper')
    return [name for name in RUNS if run_filter is None or name in run_filter]


In [ ]:
PROTOCOL_TEXT="# Bounded P3 refinement at crop1.5 + SF, v1\n\n## Scope and frozen reference\n\nUse notebook13's P3_SF_crop15 implementation as the primary base. Frozen Train10K/Val3K, data seed43/train seed100, official PAD-pretrained MiniFASNetV2/128-D/binary+11/5/40 heads, source sample order, per-sample/epoch bbox and augmentation streams, gain RNG, auxiliary losses, batch256/AMP/SGD(.005,.9,5e-4), max20/milestones[6,14]/gamma.2, one clean warm-up epoch, early-stop earliest8/patience4, Val minACER/higher-AUC selection remain unchanged. No phase-aware stopping modification. All Train/Val/LCC use crop1.5.\n\nMandatory runs:\n\n| Run | Geometry | Clean / selected-worst CE |\n|---|---|---|\n| P3_SF_W35_crop15 | SF | .65 / .35 |\n| P3_SF_W50_crop15 | SF | .50 / .50 |\n\nSF centers are [.15,.45,.75]*(1.5/2.7) = [.0833333333,.25,.4166666667]; sigma=.10*(1.5/2.7)=.0555555556. Gain Uniform(.65,.90), shared across bands/channels, preserved phase/DC and no amplification. BN-safe eval/no_grad worst selection and differentiable selected view are preserved. Aux is .1 spoof CE+.1 lighting CE+Real-only attributes BCE, clean-only. Warm-up is clean CE+aux with no spectral weighting.\n\nNotebook13 currently hard-codes .75/.25 in the loss. Notebook15 changes only this active objective to read configured coefficients. Deterministic checks verify reference scalar and backward equivalence at .75/.25, and changed scalar/gradient at W35/W50. Notebook13 itself remains unchanged.\n\nW35/W50 already exist at crop2.7 as notebook06 R3_worst_ce35/R4_worst_ce50. This round tests their interaction with crop1.5 and SF geometry, not a generic new CE-weight optimum. No additional weight, geometry, gain, crop, loss, seed or augmentation search.\n\n## DSF: mandatory source-only preflight before enabling a third run\n\nPreflight reads only frozen Train10K and physical source image headers to obtain dimensions, without detector or target data. Audit the exact per-sample training jitter at epoch2 (first spectral-active epoch): bbox_jitter(Random(BBOX_SEED+epoch*100000+sample_index)). Also record unjittered geometry for context. Preflight is not training and does not consume the global training/augmentation/gain RNG streams.\n\nThe original crop helper is instrumented to return its existing continuous `used=min((H-1)/bbox_height,(W-1)/bbox_width,1.5)`. This is s_eff relative to the actual bbox supplied after jitter, before integer rounding. Pixel-size ratios must not replace it. Its original resize result and geometry remain unchanged. Metadata exposes requested boundary interaction, actual inward window translation, and true scale reduction separately. Translation can occur while s_eff remains exactly1.5.\n\nUse header-derived shape-only arrays and the same helper's geometry-only return; no second crop geometry implementation. Save per-sample CSV with dimensions, original/jittered box, effective scales, translation and reduction flags, plus JSON count/mean/std/min/p05/median/p95/max, fraction |s_eff-1.5|>.01, fraction below1.49/below1.45, boundary/translation/reduction fractions. Both source populations have count10000. The epoch2 distribution controls the decision.\n\nPrespecified source-only gate: enable DSF iff at least1% of audited training samples deviate by more than.01 and effective-scale std>.005. These thresholds define geometric non-degeneracy, not a tuned performance threshold. They are frozen in config/report. Translation alone never enables DSF. This audit covers epoch2; later jitter draws can differ, an explicit limitation. If the gate fails, report exactly: “not run — geometry effectively degenerate under current crop helper”. Do not substitute a third experiment.\n\nCreation-time local resources do not include source image files/dimensions, so no real audit statistics or DSF-enabled claim is fabricated. Kaggle preflight runs before `P3_DSF_crop15` can be appended to enabled RUNS. Explicit DSF filtering fails clearly if preflight does not enable it. W35/W50 can run independently of DSF training, with the required audit saved in the overall artifacts.\n\nIf enabled, DSF uses ratio_i=s_eff_i/2.7, centers_i=[.15,.45,.75]*ratio_i and sigma_i=.10*ratio_i; clean/worst=.75/.25. Only training preprocessing geometry is used. Inference remains ordinary crop1.5. A separate dynamic-mask generator/selection path handles B×3×80×80 masks; the static W35/W50 generator/selection helper remains untouched. Check s_eff=1.5 geometry and all-1.5 batch equivalence to static SF, batch/band alignment, and no static-run entry into the dynamic branch.\n\n## Evaluation, resume and artifacts\n\nInside epochs infer Val3K only. Reload best, recompute/freeze Val3K minACER threshold, then infer physical LCC train/dev/eval at crop1.5 and recompute pooled metrics from concatenated scored predictions. Preserve failed candidates for coverage reporting. Raw counts8299/2948/7580/18827. No full CelebA Test, target fitting, checkpoint/early-stop selection from LCC, detector, GT fallback, harmful gate, R7 or severity.\n\nKeep original crop/CE/band/margin/drop/flip/gain diagnostics. Record DSF effective-scale diagnostics descriptively if enabled. Per-run config/init audit/best/last/history/diagnostics/freeze/metrics/predictions and independent resume retain run/config/geometry/init checks and committed best recovery. Completed outputs are exported immediately; per-run ZIP and overall ZIP include preflight and protocol, with FileLinks.\n\nOutput root: /kaggle/working/micro_P3_refine_crop15_v1/. Root artifacts include dsf_preflight_samples.csv, dsf_preflight_summary.json, implementation_sanity_checks.json, p3_refinement_comparison.csv/json, p3_refinement_report.md and this protocol. RUN_FILTER=None runs mandatory W35/W50 and DSF only if source preflight enables it; explicit subsets are supported. No historical control is retrained.\n\n## References, interpretation and bounded recommendation\n\nOptional C_crop15 and P3_SF_crop15 CSV/JSON summaries supply measured deltas across source/LCC endpoints. User-provided historical pooled AUC values .830642 and .837525 remain labeled as provided references when no full artifact is attached; missing metrics remain unavailable. Descriptive screening target >=.840642 is not a stop/search rule. Optional notebook06 summaries provide only a contextual crop2.7 W35/W50 table; they never change run definitions.\n\nDiscuss the CE-weight×crop1.5×SF interaction separately from generic crop2.7 weight behavior, and source guard/target AUC versus calibration-only HTER changes. At most one completed new variant can be suggested as a candidate for full-scale validation based on the prespecified descriptive AUC comparison; no automatic new run or final winner. LCC does not influence training/checkpoint/calibration. Only if DSF and W35 independently show useful pooled-AUC evidence versus SF, with non-worsened source ACER available, the report may flag a future P3_DSF_W35 as a conditional option; it is not implemented or executed.\n\nLimitations: Train10K/Val3K screening, one seed, repeatedly used LCC external-development benchmark, no significance claim, multi-seed robustness deferred by compute constraints, configured/realized geometry approximation and epoch2 preflight scope. No expensive local training. Validate parse/compile/cleared outputs, exact static controls/weights/warmup, scalar+backward equivalence, conditional DSF gate/masks, source-only selection/calibration and independent resume/export using tiny offline checks.\n"

import hashlib, time, importlib.util, inspect, json, math, os, random, zipfile
from collections import Counter, OrderedDict
from pathlib import Path
import albumentations as A
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader
from IPython.display import FileLink, display
from PIL import Image

AUG_SEED=TRAIN_SEED+300000
BBOX_SEED=TRAIN_SEED+500000
GAIN_SEED=TRAIN_SEED+11000
GAIN_RANGE=(.65,.90)
SPLITS=('training','development','evaluation')
EXPECTED_LCC={'training':8299,'development':2948,'evaluation':7580,'combined':18827}
validate_requested_filter(RUN_FILTER)
print('[Setup] Requested runs:',RUN_FILTER,'; DSF availability will be decided by source geometry preflight',flush=True)
if (TRAIN_CROP_FACTOR,EVAL_CROP_FACTOR)!=(1.5,1.5) or any(spec['crop_factor']!=1.5 for spec in RUNS.values()):raise ValueError('This round uses crop 1.5 only')
if NUM_WORKERS<1:raise ValueError('NUM_WORKERS must be >= 1')
if (BATCH_SIZE,MAX_EPOCHS,MILESTONES,GAMMA,TRAIN_SEED)!=(256,20,[6,14],.2,100):raise RuntimeError('Frozen micro recipe changed')
root=Path(RESOURCE_ROOT);out=Path(OUTPUT_ROOT);source=Path(MINIFASNET_SOURCE_PATH);pretrained=Path(MINIFASNET_CHECKPOINT_PATH)
for path in (source,pretrained,root/'manifests/mini_train10k.csv',root/'manifests/mini_val3k.csv',
             root/'bbox/celeba_bbox_cache_full.json',
             root/'labels/celeba_auxiliary_labels.csv'):
    if not path.is_file():raise FileNotFoundError(path)
def read_json(path):return json.loads(Path(path).read_text())
def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,sort_keys=True,allow_nan=False)+'\n')
def save_torch(path,value):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    torch.save(value,tmp);os.replace(tmp,path)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False

def state_sha(state):
    h=hashlib.sha256()
    for k in sorted(state):
        v=state[k].detach().cpu().contiguous();h.update(k.encode());h.update(str(v.dtype).encode());h.update(str(tuple(v.shape)).encode());h.update(v.numpy().tobytes())
    return h.hexdigest()

def geometry_summary(g):
    return {'number_of_evaluated_samples':len(g),'boundary_clipped_fraction':float(g[:,0].mean()),
      'mean_crop_width_over_image_width':float(g[:,1].mean()),'mean_crop_height_over_image_height':float(g[:,2].mean()),
      'median_crop_width_over_image_width':float(np.median(g[:,1])),'median_crop_height_over_image_height':float(np.median(g[:,2])),
      'effective_scale_mean':float(g[:,3].mean()),'effective_scale_std':float(g[:,3].std()),
      'window_translation_fraction':float(g[:,4].mean()),'actual_scale_reduction_fraction':float(g[:,5].mean())}

def geometry_vector(g):
    return np.asarray([g['boundary_interaction'],g['width_ratio'],g['height_ratio'],g['effective_scale'],g['window_translated'],g['scale_reduced']],dtype=np.float64)

def require_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD' or rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):raise RuntimeError('Unscorable/non-SCRFD: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Not a raw bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite(box).all() or x2<=x1 or y2<=y1:raise RuntimeError('Invalid bbox: '+key)
    return rec

def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE,return_geometry=False,geometry_only=False):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    initial_window=(left,top,right,bottom)
    requested_w,requested_h=box_w*float(scale),box_h*float(scale)
    requested_boundary=(cx-requested_w/2<0 or cy-requested_h/2<0 or cx+requested_w/2>src_w-1 or cy+requested_h/2>src_h-1)
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    geometry={'effective_scale':float(used),'window_translated':bool(any(abs(a-b)>1e-12 for a,b in zip(initial_window,(left,top,right,bottom)))),
      'scale_reduced':bool(used<float(scale)-1e-12),'boundary_interaction':bool(requested_boundary or used<float(scale)),
      'width_ratio':(rx-lx+1)/src_w,'height_ratio':(by-ty+1)/src_h}
    if geometry_only:return geometry
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    resized=cv2.resize(patch,(size,size))
    return (resized,geometry) if return_geometry else resized

# The literature's augmentation ranges are fixed. Support both documented Albumentations APIs.
# ISONoise works in RGB/HSV; convert only within augmentation, then restore BGR for MiniFASNet.
iso_params=inspect.signature(A.ISONoise).parameters
brightness_params=inspect.signature(A.RandomBrightnessContrast).parameters
blur_params=inspect.signature(A.MotionBlur).parameters
if {'color_shift','intensity'}<=set(iso_params) and {'brightness_limit','contrast_limit'}<=set(brightness_params) and 'blur_limit' in blur_params:
    AUGMENTATION_API='legacy_limits'
    iso=A.ISONoise(color_shift=(.15,.35),intensity=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_limit=.2,contrast_limit=.2,brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_limit=5,p=.2)
elif {'color_shift_range','intensity_range'}<=set(iso_params) and {'brightness_range','contrast_range'}<=set(brightness_params) and 'blur_range' in blur_params:
    AUGMENTATION_API='range_parameters'
    iso=A.ISONoise(color_shift_range=(.15,.35),intensity_range=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_range=(-.2,.2),contrast_range=(-.2,.2),brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_range=(3,5),p=.2)
else:raise RuntimeError('Unsupported Albumentations augmentation API; freeze and audit a compatible version')
COPY_AUG=A.Compose([A.HorizontalFlip(p=.5),iso,bright,motion])
def copied_aug_bgr(patch):
    rgb=cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)
    augmented=COPY_AUG(image=rgb)['image']
    return cv2.cvtColor(augmented,cv2.COLOR_RGB2BGR)
def tensor_of(patch):
    if patch.shape!=(80,80,3) or patch.dtype!=np.uint8:raise ValueError('Expected BGR uint8 80x80')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def pad_score(logits):
    z=torch.as_tensor(logits)
    if z.shape[1]==3:return (z[:,0]-torch.logsumexp(z[:,1:],dim=1)).float().cpu().numpy()
    if z.shape[1]==2:return (z[:,0]-z[:,1]).float().cpu().numpy()
    raise ValueError('Expected two or three PAD logits')
def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:raise ValueError('Val needs all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0);nr=int(real.sum());na=len(y)-nr
    bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)];rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]
def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}

def radial_masks(centers,sigma,size=INPUT_SIZE,device=None):
    if device is None:device=globals()['device']
    # Same radial geometry as the tested 03b CSMR notebook.
    y=torch.arange(size,device=device,dtype=torch.float32)-size//2
    yy,xx=torch.meshgrid(y,y,indexing='ij')
    radius=torch.sqrt(xx.square()+yy.square())/math.sqrt(2*(size//2)**2)
    centers=torch.as_tensor(centers,dtype=torch.float32,device=device)
    masks=torch.exp(-((radius[None]-centers[:,None,None])**2)/(2*sigma**2))
    if masks.shape!=(3,size,size) or not torch.isfinite(masks).all():raise RuntimeError('Invalid CSMR radial masks')
    return masks
def spectral_views(x,gains,masks):
    if x.ndim!=4 or x.shape[1:]!=(3,80,80) or not torch.isfinite(x).all():raise ValueError('CSMR input')
    x=x.float();gains=gains.float().reshape(-1)
    if len(gains)!=len(x) or not ((gains>=0)&(gains<=1)).all():raise ValueError('CSMR gain')
    spectrum=torch.fft.fftshift(torch.fft.fft2(x,norm='ortho'),dim=(-2,-1))
    fields=1-(1-gains[:,None,None,None])*masks[None]
    if CSMR_PRESERVE_DC:fields[:,:,40,40]=1
    raw=torch.fft.ifft2(torch.fft.ifftshift(spectrum[:,None]*fields[:,:,None],dim=(-2,-1)),norm='ortho').real
    clips=((raw<0)|(raw>1)).float().mean(dim=(2,3,4));views=raw.clamp(0,1)
    if not torch.isfinite(views).all():raise RuntimeError('Nonfinite CSMR views')
    return views,clips
def aligned_margin(logits,binary):
    d=logits[:,0].float()-logits[:,1].float()
    return torch.where(binary==0,d,-d)

def auxiliary_loss(outputs,binary,spoof,lighting,attrs):
    pad,spoof_logits,light_logits,attr_logits=outputs
    ls=F.cross_entropy(spoof_logits,spoof);ll=F.cross_entropy(light_logits,lighting);mask=binary==0
    la=F.binary_cross_entropy_with_logits(attr_logits[mask],attrs[mask]) if mask.any() else attr_logits.sum()*0
    return ls,ll,la
def bbox_jitter(rec,rng):
    x1,y1,x2,y2=map(float,rec['bbox_xyxy']);w,h=x2-x1,y2-y1
    if rng.random()<.20:
        base=max(w,h);cx,cy=(x1+x2)/2,(y1+y2)/2;scale=rng.uniform(.95,1.05)
        cx+=rng.uniform(-.05,.05)*base;cy+=rng.uniform(-.05,.05)*base;w*=scale;h*=scale
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]
    return [x1,y1,x2,y2]

def copied_aug_bgr_seeded(patch,seed):
    # Augmentation RNG cannot consume bbox or spectral RNG state.
    random.seed(seed);np.random.seed(seed)
    if hasattr(COPY_AUG,'set_random_seed'):COPY_AUG.set_random_seed(seed)
    return copied_aug_bgr(patch)

def load_frame(name):return pd.read_csv(root/'manifests'/(name+'.csv'),keep_default_na=False)
train=load_frame('mini_train10k');val=load_frame('mini_val3k')
if (len(train),len(val))!=(10000,3000):raise RuntimeError('Unexpected frozen CelebA populations')
if train.sample_key.duplicated().any() or val.sample_key.duplicated().any() or set(train.sample_key)&set(val.sample_key):
    raise RuntimeError('Train/Val sample overlap')
if set(train.subject_id.astype(str))&set(val.subject_id.astype(str)):raise RuntimeError('Train/Val subject overlap')
for frame in (train,val):
    if 'sampling_seed' in frame and set(frame.sampling_seed)!={DATA_SEED}:raise RuntimeError('Frozen micro seed differs')
    if not frame.usable_for_pad.astype(bool).all() or set(frame.bbox_origin)!={'SCRFD'} or set(frame.three_class_label)!={0,1,2}:
        raise RuntimeError('Frozen mini split is not fully usable')
records=read_json(root/'bbox/celeba_bbox_cache_full.json')['records']
aux=pd.read_csv(root/'labels/celeba_auxiliary_labels.csv',keep_default_na=False).set_index('sample_key',verify_integrity=True)
for frame in (train,val):
    if not set(frame.sample_key)<=set(records) or not set(frame.sample_key)<=set(aux.index):raise RuntimeError('Missing CelebA cache/aux entry')
lcc_frames={};lcc_records={}
for split in ('training','development','evaluation','combined'):
    name='lcc_official_evaluation_full' if split=='evaluation' else 'lcc_'+split+'_full'
    cache_name='lcc_official_evaluation_bbox_cache' if split=='evaluation' else 'lcc_'+split+'_bbox_cache'
    path=root/'manifests'/(name+'.csv');cache=root/'bbox'/(cache_name+'.json')
    if not path.is_file() or not cache.is_file():raise FileNotFoundError((path,cache))
    frame=pd.read_csv(path,keep_default_na=False)
    if len(frame)!=EXPECTED_LCC[split] or frame.path.duplicated().any():raise RuntimeError('Unexpected LCC population: '+split)
    recs=read_json(cache)['records'] if split!='combined' else None
    if recs is not None and set(frame.path)!=set(recs):raise RuntimeError('LCC manifest/cache keys differ: '+split)
    lcc_frames[split]=frame;lcc_records[split]=recs
ordered_pool=pd.concat([lcc_frames[s].assign(lcc_split=s) for s in SPLITS],ignore_index=True)
if not lcc_frames['combined'].equals(ordered_pool):raise RuntimeError('Combined LCC manifest differs from physical splits')
if not torch.cuda.is_available():raise RuntimeError('Kaggle CUDA GPU required for the frozen micro batch size')
source_spec=importlib.util.spec_from_file_location('local_minifasnet_v2',source)
module=importlib.util.module_from_spec(source_spec);source_spec.loader.exec_module(module)
device=torch.device('cuda');use_amp=bool(AMP)
out.mkdir(parents=True,exist_ok=True);(out/'runs').mkdir(exist_ok=True)
(out/'P3_REFINEMENT_CROP15_PROTOCOL_v1.md').write_text(PROTOCOL_TEXT)
print('GPU:',torch.cuda.get_device_name(0),'Train/Val:',len(train),len(val),'LCC:',{s:len(f) for s,f in lcc_frames.items()},flush=True)

def init_model():
    seed_all(TRAIN_SEED)
    model=PADNet(True)
    raw=torch.load(pretrained,map_location='cpu',weights_only=True)
    if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):
        raise RuntimeError('Official PAD checkpoint format differs')
    stripped=OrderedDict((k[7:],v) for k,v in raw.items())
    if tuple(stripped['prob.weight'].shape)!=(3,128) or tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):
        raise RuntimeError('Official PAD checkpoint architecture differs')
    compatible={k:v for k,v in stripped.items() if k!='prob.weight'}
    target=model.trunk.state_dict()
    if set(compatible)!=set(target) or any(tuple(compatible[k].shape)!=tuple(target[k].shape) for k in compatible):
        raise RuntimeError('PAD pretrained trunk keys differ')
    model.trunk.load_state_dict(compatible,strict=True)
    return model.to(device)

class CelebaData(Dataset):
    def __init__(self,frame,training=False,epoch=0,crop_factor=1.5):
        self.rows=frame.reset_index(drop=True);self.training=training;self.epoch=epoch;self.crop_factor=crop_factor
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.sample_key);rec=require_record(records[key],key)
        image=cv2.imread(str(Path(CELEBA_ROOT)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(key)
        box=bbox_jitter(rec,random.Random(BBOX_SEED+self.epoch*100000+i)) if self.training else rec['bbox_xyxy']
        patch,metadata=mini_crop_bgr(image,box,scale=self.crop_factor,return_geometry=True)
        geometry=geometry_vector(metadata)
        if self.training:patch=copied_aug_bgr_seeded(patch,AUG_SEED+self.epoch*100000+i)
        a=aux.loc[key]
        return tensor_of(patch),int(row.three_class_label),int(row.binary_label),int(a.spoof_type),int(a.lighting),torch.tensor([float(a[f'attr_{j:02d}']) for j in range(40)]),key,geometry

class LCCData(Dataset):
    def __init__(self,split,crop_factor):
        self.split=split;self.crop_factor=crop_factor;self.rows=lcc_frames[split].loc[lcc_frames[split].usable_for_pad.astype(bool)].reset_index(drop=True)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.path);rec=require_record(lcc_records[self.split][key],key)
        path=(Path(LCC_ROOT)/'LCC_FASD_evaluation'/key) if self.split=='evaluation' else Path(LCC_ROOT)/key
        image=cv2.imread(str(path),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(path)
        patch,metadata=mini_crop_bgr(image,rec['bbox_xyxy'],scale=self.crop_factor,return_geometry=True)
        return tensor_of(patch),int(row.label),key,geometry_vector(metadata)

def loader(ds,shuffle=False,epoch=0):
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=True,
                      drop_last=False,generator=gen)

def infer(model,dl,lcc=False,label='Evaluation'):
    model.eval();keys=[];ys=[];zs=[];geometry=[];started=time.monotonic()
    print('['+label+'] Starting',len(dl.dataset),'scored images',flush=True)
    with torch.no_grad():
        for step,batch in enumerate(dl,1):
            if lcc:x,y,k,g=batch
            else:x,y,_,_,_,_,k,g=batch
            with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))
            keys.extend(k);ys.extend(y.numpy().tolist());zs.append(z.float().cpu().numpy());geometry.append(g.numpy())
            if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(dl):
                print(f'[{label}] batch {step}/{len(dl)}, elapsed {time.monotonic()-started:.1f}s',flush=True)
    if not zs:raise RuntimeError('No scorable samples')
    return list(map(str,keys)),np.asarray(ys,dtype=int),np.concatenate(zs),np.concatenate(geometry)

def select_worst_view(model,base,binary,masks,gain_rng):
    was_training=model.training;model.eval()
    chosen=[];bands=[];gains_out=[];clips_out=[];clean_margins=[];worst_margins=[]
    try:
        with torch.no_grad():
            all_gains=torch.empty(len(base),device=base.device).uniform_(*GAIN_RANGE,generator=gain_rng)
            for start in range(0,len(base),SELECTION_CHUNK_SIZE):
                x=base[start:start+SELECTION_CHUNK_SIZE];y=binary[start:start+SELECTION_CHUNK_SIZE]
                gains=all_gains[start:start+len(x)]
                with torch.cuda.amp.autocast(enabled=use_amp):clean_z=model(x)
                clean_margins.append(aligned_margin(clean_z,y).detach())
                views,clips=spectral_views(x,gains,masks);margins=[]
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=use_amp):z=model(views[:,band])
                    margins.append(aligned_margin(z,y))
                margin=torch.stack(margins,dim=1);ix=margin.argmin(dim=1)
                row=torch.arange(len(x),device=x.device)
                chosen.append(views[row,ix].detach());bands.append(ix.detach())
                gains_out.append(gains.detach());clips_out.append(clips[row,ix].detach())
                worst_margins.append(margin[row,ix].detach())
    finally:model.train(was_training)
    return tuple(torch.cat(seq) for seq in (chosen,bands,gains_out,clips_out,clean_margins,worst_margins))


def training_terms(model,x,binary,spoof,lighting,attrs,spec,masks,gain_rng,active,effective_scales=None):
    n=len(x);selected=None;bands=None;gains=None;select_clean=None;select_worst=None
    if active:
        if spec['geometry_mode']=='DYNAMIC':
            if effective_scales is None:raise ValueError('DSF needs geometry from the actual training crop')
            selected,bands,gains,_,select_clean,select_worst=select_worst_view_dynamic(model,x,binary,effective_scales,gain_rng)
        else:
            selected,bands,gains,_,select_clean,select_worst=select_worst_view(model,x,binary,masks,gain_rng)
    with torch.cuda.amp.autocast(enabled=use_amp):
        clean=model(x,aux=True);clean_ce=F.cross_entropy(clean[0],binary)
        ls,ll,la=auxiliary_loss(clean,binary,spoof,lighting,attrs);aux_total=.1*ls+.1*ll+la
        worst_ce=clean_ce*0;spectral_logits=None
        if active:
            spectral_logits=model(selected);worst_ce=F.cross_entropy(spectral_logits,binary)
        loss=p3_objective(clean_ce,worst_ce,aux_total,spec,active)
    with torch.no_grad():
        cm=aligned_margin(clean[0],binary).mean()
        flips=int(((clean[0].argmax(1)==binary)&(spectral_logits.argmax(1)!=binary)).sum()) if spectral_logits is not None else 0
        scalars={'train_total_loss':float(loss.detach()),'clean_ce':float(clean_ce.detach()),'spectral_ce':float(worst_ce.detach()),
          'worst_ce':float(worst_ce.detach()),'aux_loss':float(aux_total.detach()),
          'clean_aligned_margin':float(select_clean.mean()) if select_clean is not None else float(cm),
          'train_clean_aligned_margin':float(cm),'selection_clean_margin':float(select_clean.mean()) if select_clean is not None else 0.,
          'worst_margin':float(select_worst.mean()) if select_worst is not None else 0.,
          'margin_drop':float((select_clean-select_worst).mean()) if select_worst is not None else 0.,
          'spectral_flip_rate':flips/n,'sampled_gain_mean':float(gains.mean()) if gains is not None else 0.}
    counts=Counter(bands.detach().cpu().tolist()) if bands is not None else Counter()
    return loss,scalars,counts

def snapshot_rng(gain_rng):
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
      'cuda':torch.cuda.get_rng_state_all(),'gain':gain_rng.get_state()}

def restore_rng(state,gain_rng):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda']);gain_rng.set_state(state['gain'])

def run_config(name,spec):
    return {'run_id':name,'spec':spec,'resource_root':str(root),'train_manifest':'mini_train10k.csv','val_manifest':'mini_val3k.csv',
      'data_seed':DATA_SEED,'train_seed':TRAIN_SEED,'train_crop_factor':spec['crop_factor'],'eval_crop_factor':spec['crop_factor'],
      'base_order':'torch Generator(TRAIN_SEED+epoch)','bbox_seed':BBOX_SEED,'augmentation_seed':AUG_SEED,'gain_seed':GAIN_SEED,
      'pretrained_checkpoint':str(pretrained),'model_source':str(source),'celeba_root':CELEBA_ROOT,'lcc_root':LCC_ROOT,
      'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'milestones':MILESTONES,'gamma':GAMMA,'num_workers':NUM_WORKERS,
      'optimizer':{'name':'SGD','lr':.005,'momentum':.9,'weight_decay':5e-4},
      'warmup_epochs':0 if spec['method']=='CLEAN' else WARMUP_EPOCHS,
      'earliest_early_stop_epoch':EARLIEST_EARLY_STOP_EPOCH,'patience':EARLY_STOP_PATIENCE,
      'spectral':{'centers':spec['centers'],'sigma':spec['sigma'],'gain':[.65,.90],'preserve_dc':CSMR_PRESERVE_DC,'geometry_mode':spec['geometry_mode'],'reference_crop':REFERENCE_CROP},
      'input':'80x80 BGR float [0,1]','augmentation_api':AUGMENTATION_API,'albumentations_version':A.__version__,'amp':use_amp}

def prediction_frame(frame,keys,y,z,threshold,keycol):
    scores=pad_score(z)
    scored=pd.DataFrame({keycol:keys,'pad_logit_score':scores,'pred_real':scores>=threshold,
                         'logit_0':z[:,0],'logit_1':z[:,1]})
    result=frame.merge(scored,on=keycol,how='left',validate='one_to_one',sort=False)
    valid=frame.usable_for_pad.astype(bool)
    if result.loc[valid,'pad_logit_score'].isna().any() or result.loc[~valid,'pad_logit_score'].notna().any():
        raise RuntimeError('Prediction accounting differs from frozen candidates')
    result['locked_threshold']=threshold
    return result

def evaluate_lcc(model,folder,locked,crop_factor,crop_diagnostics):
    scored_predictions=[];all_geometry=[]
    for split in SPLITS:
        frame=lcc_frames[split]
        keys,y,z,g=infer(model,loader(LCCData(split,crop_factor)),True,label=folder.name+' LCC '+split)
        crop_diagnostics['lcc_'+split]=geometry_summary(g);all_geometry.append(g)
        expected=frame.loc[frame.usable_for_pad.astype(bool),'path'].astype(str).tolist()
        if keys!=expected:raise RuntimeError('LCC scored order differs: '+split)
        pred=prediction_frame(frame,keys,y,z,locked,'path').assign(lcc_split=split)
        mm=metrics(y,pad_score(z),locked,len(frame))
        write_json(folder/f'lcc_{split}_metrics.json',mm)
        pred.to_csv(folder/'predictions'/f'lcc_{split}_predictions.csv',index=False)
        scored_predictions.append(pred)
    pooled=pd.concat(scored_predictions,ignore_index=True)
    if not pooled[['path','label','status','usable_for_pad','bbox_origin','lcc_split']].equals(
        lcc_frames['combined'][['path','label','status','usable_for_pad','bbox_origin','lcc_split']]):
        raise RuntimeError('Combined prediction candidates differ from 00d manifest')
    usable=pooled.usable_for_pad.astype(bool)
    mm=metrics(pooled.loc[usable,'label'],pooled.loc[usable,'pad_logit_score'],locked,len(pooled))
    write_json(folder/'lcc_combined_metrics.json',mm)
    pooled.to_csv(folder/'predictions'/'lcc_combined_predictions.csv',index=False)
    crop_diagnostics['lcc_combined']=geometry_summary(np.concatenate(all_geometry))


def run_one(name,spec):
    folder=out/'runs'/name;folder.mkdir(parents=True,exist_ok=True);(folder/'predictions').mkdir(exist_ok=True)
    print('['+name+'] Starting matched crop',spec['crop_factor'],flush=True)
    cfg=run_config(name,spec);cfg_path=folder/'config.json'
    if cfg_path.is_file() and read_json(cfg_path)!=cfg:raise RuntimeError(name+' existing config differs')
    write_json(cfg_path,cfg)
    best_path=folder/'best.pth';last_path=folder/'last_state.pth';done_path=folder/'completion.json'
    if done_path.is_file():
        needed=[folder/(k+'_metrics.json') for k in ('val3k','lcc_training','lcc_development','lcc_evaluation','lcc_combined')]
        if not all(p.is_file() for p in needed) or not best_path.is_file():
            raise RuntimeError(name+' incomplete completion outputs')
        print(name,'completed; skipping',flush=True)
        return
    model=init_model()
    initial_sha=state_sha(model.state_dict())
    write_json(folder/'initialization_audit.json',{'initial_state_sha256':initial_sha,'train_seed':TRAIN_SEED,'pretrained_checkpoint':str(pretrained)})
    optimizer=torch.optim.SGD(model.parameters(),lr=.005,momentum=.9,weight_decay=5e-4)
    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=MILESTONES,gamma=GAMMA)
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    gain_rng=torch.Generator(device=device).manual_seed(GAIN_SEED)
    best_checkpoint=None;crop_diagnostics={'training_epochs':{}}
    history=[];diagnostics=[];best_acer=float('inf');best_auc=-float('inf');best_epoch=0;no_improve=0;start=1
    if last_path.is_file():
        if not RESUME:raise RuntimeError(name+' has a saved state; enable RESUME or use a new output root')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if state.get('run_id')!=name or state.get('config')!=cfg:raise RuntimeError(name+' resume run/config differs')
        if state['initial_state_sha256']!=initial_sha:raise RuntimeError('Resume initial state differs')
        best_checkpoint=state['best_checkpoint'];crop_diagnostics=state['crop_diagnostics']
        if state['best_epoch']:
            if best_checkpoint['run_id']!=name or best_checkpoint['epoch']!=state['best_epoch'] or best_checkpoint['config']!=cfg:raise RuntimeError('Resume committed best differs')
            save_torch(best_path,best_checkpoint)
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        history=state['history'];diagnostics=state['diagnostics'];best_acer=state['best_acer']
        best_auc=state['best_auc'];best_epoch=state['best_epoch'];no_improve=state['no_improve']
        start=state['epoch']+1
        if len(history)!=state['epoch']:raise RuntimeError(name+' resume history length differs')
        restore_rng(state['rng'],gain_rng)
        print(name,'resume at epoch',start,flush=True)
    val_dl=loader(CelebaData(val,crop_factor=spec['crop_factor']))
    masks=radial_masks(spec['centers'],spec['sigma']) if spec['geometry_mode']=='STATIC' else None
    if start<=MAX_EPOCHS and not (history and history[-1]['stop_reached']):
        for epoch in range(start,MAX_EPOCHS+1):
            warmup=spec['method']!='CLEAN' and epoch<=WARMUP_EPOCHS
            active=not warmup
            model.train();totals=Counter();band_count=Counter();seen=0;train_geometry=[];
            train_ds=CelebaData(train,training=True,epoch=epoch,crop_factor=spec['crop_factor'])
            train_dl=loader(train_ds,shuffle=True,epoch=epoch);started=time.monotonic()
            print(f'[{name}] epoch {epoch}/{MAX_EPOCHS} starting, phase={"warmup" if warmup else "active"}, lr={optimizer.param_groups[0]["lr"]:g}',flush=True)
            for step,(x,_,binary,spoof,lighting,attrs,_,g) in enumerate(train_dl,1):
                train_geometry.append(g.numpy())
                x=x.to(device,non_blocking=True);binary=binary.to(device);spoof=spoof.to(device)
                lighting=lighting.to(device);attrs=attrs.to(device);optimizer.zero_grad(set_to_none=True)
                loss,scalars,bands=training_terms(model,x,binary,spoof,lighting,attrs,
                    spec,masks,gain_rng,active,effective_scales=g[:,3].to(device) if spec['geometry_mode']=='DYNAMIC' else None)
                scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
                count=len(x);seen+=count
                for key,value in scalars.items():totals[key]+=value*count
                band_count.update(bands)
                if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(train_dl):
                    print(f'[{name}] epoch {epoch} batch {step}/{len(train_dl)}, loss={totals["train_total_loss"]/seen:.4f}, elapsed={time.monotonic()-started:.1f}s',flush=True)
            crop_diagnostics['training_epochs'][str(epoch)]=geometry_summary(np.concatenate(train_geometry))
            keys,y,z,g=infer(model,val_dl,label=f'{name} epoch {epoch} Val3K')
            if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Val order differs')
            scores=pad_score(z);threshold=threshold_from_val(y,scores)
            vm=metrics(y,scores,threshold,len(val))
            improved=active and (vm['acer']<best_acer-METRIC_TIE_EPS or
                (abs(vm['acer']-best_acer)<=METRIC_TIE_EPS and vm['auc']>best_auc+METRIC_TIE_EPS))
            if active:
                if improved:
                    best_acer=vm['acer'];best_auc=vm['auc'];best_epoch=epoch;no_improve=0
                    best_checkpoint={'run_id':name,'epoch':epoch,'config':cfg,
                        'val_acer':best_acer,'val_auc':best_auc,
                        'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}}
                    save_torch(best_path,best_checkpoint)
                else:no_improve+=1
            row={'epoch':epoch,'phase':'warmup' if warmup else 'active','lr':optimizer.param_groups[0]['lr'],
                 'eligible_for_final_checkpoint':active,'val_acer':vm['acer'],'val_auc':vm['auc'],
                 'val_eer':vm['eer'],'val_threshold':threshold,'best_epoch':best_epoch,
                 'no_improve_count':no_improve,'clean_ce_weight':1.0 if warmup else spec['clean_ce_weight'],
                 'spectral_ce_weight':0.0 if warmup else spec['spectral_ce_weight'],
                 'method':'CLEAN_WARMUP' if warmup else spec['method'],
                 **{k:v/seen for k,v in totals.items()},
                 **{f'{label}_count':band_count[i] for i,label in enumerate(('low','mid','high'))},
                 **{f'{label}_fraction':band_count[i]/seen for i,label in enumerate(('low','mid','high'))}}
            diagnostics.append(dict(row))
            stop=epoch==MAX_EPOCHS or (active and epoch>=EARLIEST_EARLY_STOP_EPOCH and no_improve>=EARLY_STOP_PATIENCE)
            row['stop_reached']=stop;history.append(row)
            scheduler.step()
            state={'run_id':name,'config':cfg,'epoch':epoch,
                'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                'scaler':scaler.state_dict(),'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                'no_improve':no_improve,'rng':snapshot_rng(gain_rng),'initial_state_sha256':initial_sha,
                'best_checkpoint':best_checkpoint,'crop_diagnostics':crop_diagnostics,
                'history':history,'diagnostics':diagnostics}
            save_torch(last_path,state)
            pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
            pd.DataFrame(diagnostics).to_csv(folder/'spectral_diagnostics.csv',index=False)
            write_json(folder/'crop_diagnostics.json',crop_diagnostics)
            print(name,'epoch',epoch,'Val ACER',vm['acer'],'best',best_epoch,'patience',no_improve,flush=True)
            if stop:break
    if not best_path.is_file() or best_epoch<= (0 if spec['method']=='CLEAN' else WARMUP_EPOCHS):
        raise RuntimeError(name+' has no eligible best checkpoint')
    best=torch.load(best_path,map_location='cpu',weights_only=True)
    if best['run_id']!=name or best['config']!=cfg or best['epoch']!=best_epoch:
        raise RuntimeError(name+' best checkpoint identity differs')
    model.load_state_dict(best['state_dict']);model.eval()
    keys,y,z,g=infer(model,val_dl,label=name+' frozen Val3K')
    crop_diagnostics['val3k']=geometry_summary(g)
    if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Frozen Val order differs')
    locked=threshold_from_val(y,pad_score(z))
    vm=metrics(y,pad_score(z),locked,len(val))
    if abs(vm['acer']-best['val_acer'])>1e-6 or abs(vm['auc']-best['val_auc'])>1e-6:
        raise RuntimeError('Best checkpoint does not reproduce Val selection metrics')
    write_json(folder/'val3k_metrics.json',vm)
    prediction_frame(val,keys,y,z,locked,'sample_key').to_csv(folder/'predictions'/'val3k_predictions.csv',index=False)
    freeze={'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,'val3k_metrics':vm,
      'train_crop_factor':spec['crop_factor'],'eval_crop_factor':spec['crop_factor'],'threshold_source':'frozen Val3K at matched crop'}
    freeze_path=folder/'evaluation_freeze.json'
    if freeze_path.is_file() and read_json(freeze_path)!=freeze:raise RuntimeError('Evaluation freeze differs')
    write_json(freeze_path,freeze)
    evaluate_lcc(model,folder,locked,spec['crop_factor'],crop_diagnostics)
    write_json(folder/'crop_diagnostics.json',crop_diagnostics)
    write_json(done_path,{'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,
                          'crop_factor':spec['crop_factor'],'initial_state_sha256':initial_sha})
    del model
    torch.cuda.empty_cache()

def p3_objective(clean_ce,worst_ce,aux_total,spec,active):
    if not active:return clean_ce+aux_total
    return spec['clean_ce_weight']*clean_ce+spec['spectral_ce_weight']*worst_ce+aux_total

def coefficient_sanity_checks():
    results={}
    for label,a,b in [('reference',.75,.25),('W35',.65,.35),('W50',.50,.50)]:
        clean=torch.tensor(1.2,dtype=torch.float64,requires_grad=True)
        worst=torch.tensor(2.7,dtype=torch.float64,requires_grad=True)
        aux=torch.tensor(.4,dtype=torch.float64,requires_grad=True)
        actual=p3_objective(clean,worst,aux,{'clean_ce_weight':a,'spectral_ce_weight':b},True)
        expected=a*clean+b*worst+aux
        grads=torch.autograd.grad(actual,(clean,worst,aux),retain_graph=True)
        expected_grads=torch.autograd.grad(expected,(clean,worst,aux))
        if not torch.equal(actual,expected) or any(not torch.equal(x,y) for x,y in zip(grads,expected_grads)):raise RuntimeError('P3 coefficient scalar/backward check failed')
        results[label]={'total':float(actual.detach()),'gradients':[float(g) for g in grads]}
    if len({r['total'] for r in results.values()})!=3:raise RuntimeError('Weights did not change objective')
    clean=torch.tensor(1.2,requires_grad=True);worst=torch.tensor(2.7,requires_grad=True);aux=torch.tensor(.4,requires_grad=True)
    warm=p3_objective(clean,worst,aux,{'clean_ce_weight':.5,'spectral_ce_weight':.5},False)
    gc,gw,ga=torch.autograd.grad(warm,(clean,worst,aux),allow_unused=True)
    if float(gc)!=1 or gw is not None or float(ga)!=1:raise RuntimeError('Warm-up objective changed')
    return {'coefficient_checks':results,'reference_075_025_scalar_and_backward_equivalent':True,'warmup_clean_only':True}

def dynamic_radial_masks(effective_scales,size=INPUT_SIZE,device=None):
    device=effective_scales.device if device is None else device
    scales=torch.as_tensor(effective_scales,dtype=torch.float32,device=device).reshape(-1)
    if not len(scales) or not torch.isfinite(scales).all() or not (scales>0).all() or not (scales<=1.5+1e-6).all():raise ValueError('Invalid source effective crop scales')
    y=torch.arange(size,device=device,dtype=torch.float32)-size//2
    yy,xx=torch.meshgrid(y,y,indexing='ij');radius=torch.sqrt(xx.square()+yy.square())/math.sqrt(2*(size//2)**2)
    ratio=scales/REFERENCE_CROP
    centers=torch.as_tensor(ORIGINAL_CENTERS,dtype=torch.float32,device=device)[None]*ratio[:,None]
    sigma=REFERENCE_SIGMA*ratio
    masks=torch.exp(-((radius[None,None]-centers[:,:,None,None])**2)/(2*sigma[:,None,None,None]**2))
    if masks.shape!=(len(scales),3,size,size) or not torch.isfinite(masks).all():raise RuntimeError('Dynamic mask alignment differs')
    return masks

def dynamic_spectral_views(x,gains,masks):
    if masks.shape!=(len(x),3,INPUT_SIZE,INPUT_SIZE):raise ValueError('Dynamic masks must align batch/band/spatial dimensions')
    if x.ndim!=4 or x.shape[1:]!=(3,80,80) or not torch.isfinite(x).all():raise ValueError('CSMR input')
    x=x.float();gains=gains.float().reshape(-1)
    if len(gains)!=len(x) or not ((gains>=0)&(gains<=1)).all():raise ValueError('CSMR gain')
    spectrum=torch.fft.fftshift(torch.fft.fft2(x,norm='ortho'),dim=(-2,-1))
    fields=1-(1-gains[:,None,None,None])*masks
    if CSMR_PRESERVE_DC:fields[:,:,40,40]=1
    raw=torch.fft.ifft2(torch.fft.ifftshift(spectrum[:,None]*fields[:,:,None],dim=(-2,-1)),norm='ortho').real
    clips=((raw<0)|(raw>1)).float().mean(dim=(2,3,4));views=raw.clamp(0,1)
    if not torch.isfinite(views).all():raise RuntimeError('Nonfinite CSMR views')
    return views,clips

def select_worst_view_dynamic(model,base,binary,effective_scales,gain_rng):
    was_training=model.training;model.eval()
    chosen=[];bands=[];gains_out=[];clips_out=[];clean_margins=[];worst_margins=[]
    try:
        with torch.no_grad():
            all_gains=torch.empty(len(base),device=base.device).uniform_(*GAIN_RANGE,generator=gain_rng)
            for start in range(0,len(base),SELECTION_CHUNK_SIZE):
                x=base[start:start+SELECTION_CHUNK_SIZE];y=binary[start:start+SELECTION_CHUNK_SIZE]
                gains=all_gains[start:start+len(x)]
                masks=dynamic_radial_masks(effective_scales[start:start+len(x)],device=x.device)
                with torch.cuda.amp.autocast(enabled=use_amp):clean_z=model(x)
                clean_margins.append(aligned_margin(clean_z,y).detach())
                views,clips=dynamic_spectral_views(x,gains,masks);margins=[]
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=use_amp):z=model(views[:,band])
                    margins.append(aligned_margin(z,y))
                margin=torch.stack(margins,dim=1);ix=margin.argmin(dim=1);row=torch.arange(len(x),device=x.device)
                chosen.append(views[row,ix].detach());bands.append(ix.detach());gains_out.append(gains.detach())
                clips_out.append(clips[row,ix].detach());worst_margins.append(margin[row,ix].detach())
    finally:model.train(was_training)
    return tuple(torch.cat(seq) for seq in (chosen,bands,gains_out,clips_out,clean_margins,worst_margins))

def dynamic_mask_sanity_checks():
    scales=torch.full((2,),1.5);static=radial_masks(SCALED_CENTERS,SCALED_SIGMA,device=torch.device('cpu'))
    dynamic=dynamic_radial_masks(scales)
    if not torch.allclose(dynamic,static[None].expand(2,-1,-1,-1),atol=2e-6,rtol=1e-5):raise RuntimeError('DSF at 1.5 differs from static SF masks')
    ratios=scales/REFERENCE_CROP
    if not torch.allclose(torch.tensor(ORIGINAL_CENTERS)[None]*ratios[:,None],torch.tensor(SCALED_CENTERS)[None],atol=1e-7):raise RuntimeError('DSF center equivalence failed')
    if not torch.allclose(REFERENCE_SIGMA*ratios,torch.full_like(ratios,SCALED_SIGMA),atol=1e-7):raise RuntimeError('DSF sigma equivalence failed')
    different=dynamic_radial_masks(torch.tensor([1.5,1.2]))
    if different.shape!=(2,3,80,80) or torch.allclose(different[0],different[1]):raise RuntimeError('DSF batch alignment failed')
    return {'dsf_at_15_matches_sf_centers_sigma':True,'dsf_all_15_masks_match_static_sf':True,'dynamic_mask_shape':[2,3,80,80],'batch_alignment_checked':True}

class GeometryPreflightData(Dataset):
    def __init__(self,frame):self.keys=frame.sample_key.astype(str).tolist()
    def __len__(self):return len(self.keys)
    def __getitem__(self,i):
        key=self.keys[i];rec=require_record(records[key],key);path=Path(CELEBA_ROOT)/key
        with Image.open(path) as image:
            width,height=image.size;orientation=image.getexif().get(274,1)
            if orientation in (5,6,7,8):width,height=height,width  # Match OpenCV's default EXIF-aware image dimensions.
        # Shape only; the same crop helper returns geometry before reading/resizing pixels.
        shape_image=np.broadcast_to(np.zeros((1,1,3),dtype=np.uint8),(height,width,3))
        original=rec['bbox_xyxy'];jittered=bbox_jitter(rec,random.Random(BBOX_SEED+PREFLIGHT_EPOCH*100000+i))
        raw=mini_crop_bgr(shape_image,original,scale=1.5,geometry_only=True)
        actual=mini_crop_bgr(shape_image,jittered,scale=1.5,geometry_only=True)
        result={'sample_key':key,'image_width':width,'image_height':height,'epoch':PREFLIGHT_EPOCH,
          's_eff':actual['effective_scale'],'boundary_interaction':actual['boundary_interaction'],
          'window_translated':actual['window_translated'],'scale_reduced':actual['scale_reduced'],
          'raw_s_eff':raw['effective_scale'],'raw_boundary_interaction':raw['boundary_interaction'],
          'raw_window_translated':raw['window_translated'],'raw_scale_reduced':raw['scale_reduced']}
        for j,label in enumerate(('x1','y1','x2','y2')):result['bbox_'+label]=float(original[j]);result['jittered_bbox_'+label]=float(jittered[j])
        return result

def scale_distribution(frame,column,boundary_column,translation_column,reduction_column):
    values=frame[column].to_numpy(dtype=float)
    return {'count':len(values),'mean':float(values.mean()),'std':float(values.std()),'min':float(values.min()),
      'p05':float(np.quantile(values,.05)),'median':float(np.median(values)),'p95':float(np.quantile(values,.95)),'max':float(values.max()),
      'fraction_deviation_gt_tolerance':float((np.abs(values-1.5)>DSF_SCALE_TOLERANCE).mean()),
      'fraction_below_149':float((values<1.49).mean()),'fraction_below_145':float((values<1.45).mean()),
      'boundary_interaction_fraction':float(frame[boundary_column].mean()),
      'window_translation_fraction':float(frame[translation_column].mean()),
      'actual_scale_reduction_fraction':float(frame[reduction_column].mean())}

def geometry_gate(summary):
    return summary['fraction_deviation_gt_tolerance']>=DSF_MIN_VARIATION_FRACTION and summary['std']>DSF_MIN_SCALE_STD

def run_geometry_preflight():
    print('[DSF preflight] Auditing Train10K headers and exact epoch',PREFLIGHT_EPOCH,'jitter/crop geometry...',flush=True)
    dataset=GeometryPreflightData(train);generator=torch.Generator().manual_seed(TRAIN_SEED+PREFLIGHT_EPOCH)
    dl=DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,generator=generator)
    batches=[];started=time.monotonic()
    for step,batch in enumerate(dl,1):
        batches.append(pd.DataFrame({key:value.tolist() if torch.is_tensor(value) else list(value) for key,value in batch.items()}))
        if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(dl):print(f'[DSF preflight] batch {step}/{len(dl)}, elapsed {time.monotonic()-started:.1f}s',flush=True)
    frame=pd.concat(batches,ignore_index=True)
    if frame.sample_key.astype(str).tolist()!=train.sample_key.astype(str).tolist():raise RuntimeError('Preflight source population/order differs')
    frame.to_csv(out/'dsf_preflight_samples.csv',index=False)
    jittered=scale_distribution(frame,'s_eff','boundary_interaction','window_translated','scale_reduced')
    raw=scale_distribution(frame,'raw_s_eff','raw_boundary_interaction','raw_window_translated','raw_scale_reduced')
    enabled=geometry_gate(jittered)
    summary={'source':'frozen Train10K only','data_seed':DATA_SEED,'train_seed':TRAIN_SEED,'audited_training_epoch':PREFLIGHT_EPOCH,
      'effective_scale_definition':'crop helper used=min((H-1)/bbox_height,(W-1)/bbox_width,1.5), after exact epoch/sample jitter, before pixel rounding',
      'header_only_no_decode':True,'jittered':jittered,'unjittered':raw,
      'gate':{'scale_tolerance':DSF_SCALE_TOLERANCE,'minimum_deviation_fraction':DSF_MIN_VARIATION_FRACTION,'minimum_std':DSF_MIN_SCALE_STD},
      'dsf_enabled':bool(enabled),'dsf_status':'enabled after source preflight' if enabled else 'not run — geometry effectively degenerate under current crop helper',
      'limitation':'epoch2 jitter audit; later epoch draws can differ; boundary translation alone is not scale reduction'}
    write_json(out/'dsf_preflight_summary.json',summary)
    print('[DSF preflight]',json.dumps(summary,indent=2),flush=True)
    return summary


## Source-only DSF preflight and implementation sanity checks

Run before enabling any DSF training. No LCC values enter this decision.

In [ ]:
implementation_checks=coefficient_sanity_checks()
implementation_checks.update(dynamic_mask_sanity_checks())
implementation_checks['static_runs_use_only_static_mask_path']=all(spec['geometry_mode']=='STATIC' for spec in RUNS.values())
write_json(out/'implementation_sanity_checks.json',implementation_checks)
DSF_PREFLIGHT=run_geometry_preflight()
RUNS.pop('P3_DSF_crop15',None)
if DSF_PREFLIGHT['dsf_enabled']:RUNS['P3_DSF_crop15']=dict(DSF_SPEC)
selected=select_runs(RUN_FILTER)
print('[Preflight] Enabled runs:',list(RUNS),'Selected:',selected,flush=True)
# Inspect dsf_preflight_summary.json here before running the next cell.


## Preflight report / exports

This cell saves the audit and pending comparison before training.

In [ ]:
METRIC_COLUMNS=('val3k_acer','val3k_auc','lcc_evaluation_auc','lcc_evaluation_hter','lcc_combined_auc',
 'lcc_combined_eer','lcc_combined_hter','lcc_combined_tpr_at_fpr_1pct')
DELTA_COLUMNS=('val3k_acer','lcc_evaluation_auc','lcc_combined_auc','lcc_combined_eer','lcc_combined_hter','lcc_combined_tpr_at_fpr_1pct')

def summary_rows(path):
    path=Path(path)
    if path.suffix.lower()=='.json':
        payload=read_json(path)
        return payload if isinstance(payload,list) else payload.get('rows',[payload])
    return pd.read_csv(path,keep_default_na=False).to_dict('records')

def historical_reference(name,path):
    row={'run_id':name,'status':'provided_AUC_only','crop_factor':1.5,'reference_origin':'user-provided prompt value; not remeasured',
      'lcc_combined_auc':PROVIDED_REFERENCE_AUC[name]};notes=[]
    if path is None:return row,notes
    if not Path(path).is_file():return row,['Historical reference path unavailable: '+str(path)]
    try:
        matches=[r for r in summary_rows(path) if r.get('run_id',r.get('model'))==name]
        if len(matches)!=1:return row,[str(path)+' must contain exactly one '+name+' row; retaining supplied AUC-only reference.']
        raw=matches[0]
        if 'crop_factor' in raw and float(raw['crop_factor'])!=1.5:return row,[name+' historical row is not crop1.5; skipped.']
        row.update({'status':'historical_artifact','reference_origin':str(path)})
        for key,value in raw.items():
            target='val3k_'+key[4:] if key.startswith('val_') else key
            if target.startswith(('val3k_','lcc_')):
                try:number=float(value)
                except (ValueError,TypeError):continue
                if np.isfinite(number):row[target]=number
        if abs(row['lcc_combined_auc']-PROVIDED_REFERENCE_AUC[name])>1e-6:notes.append(name+' artifact pooled AUC differs from prompt value; table uses the attached artifact.')
    except (OSError,ValueError,TypeError,KeyError,AttributeError) as exc:notes.append(name+' reference skipped: '+str(exc))
    return row,notes

def metric_table(rows):
    lines=['| Run | Status | Clean / Worst | Geometry | '+' | '.join(METRIC_COLUMNS)+' |','|---|---|---|---|'+'---:|'*len(METRIC_COLUMNS)]
    for row in rows:
        weights=f'{row["clean_ce_weight"]:.2f} / {row["spectral_ce_weight"]:.2f}' if 'clean_ce_weight' in row else 'reference'
        lines.append('| '+row['run_id']+' | '+row['status']+' | '+weights+' | '+row.get('geometry_mode','historical')+' | '+' | '.join(
          f'{row[k]:.6f}' if isinstance(row.get(k),(int,float)) else 'pending' for k in METRIC_COLUMNS)+' |')
    return lines

def contextual_crop27_rows():
    if HISTORICAL_CROP27_WEIGHT_SUMMARY is None or not Path(HISTORICAL_CROP27_WEIGHT_SUMMARY).is_file():return []
    result=[]
    try:
        for raw in summary_rows(HISTORICAL_CROP27_WEIGHT_SUMMARY):
            name=raw.get('run_id',raw.get('model'))
            if name not in ('R1_p3_ce25','R3_worst_ce35','R4_worst_ce50'):continue
            row={'run_id':name,'status':'historical_crop27_context','crop_factor':2.7,'geometry_mode':'original fixed crop2.7',
              'clean_ce_weight':{'R1_p3_ce25':.75,'R3_worst_ce35':.65,'R4_worst_ce50':.5}[name],
              'spectral_ce_weight':{'R1_p3_ce25':.25,'R3_worst_ce35':.35,'R4_worst_ce50':.5}[name]}
            for key,value in raw.items():
                target='val3k_'+key[4:] if key.startswith('val_') else key
                if target.startswith(('val3k_','lcc_')):
                    try:number=float(value)
                    except (TypeError,ValueError):continue
                    if np.isfinite(number):row[target]=number
            result.append(row)
    except (OSError,ValueError,TypeError,KeyError,AttributeError) as exc:print('Optional crop2.7 context skipped:',str(exc),flush=True)
    return result

def export_report():
    rows=[]
    for name,spec in RUNS.items():
        folder=out/'runs'/name
        row={'run_id':name,'status':'pending','crop_factor':1.5,'geometry_mode':spec['geometry_mode'],
          'centers':spec['centers'],'sigma':spec['sigma'],'clean_ce_weight':spec['clean_ce_weight'],'spectral_ce_weight':spec['spectral_ce_weight']}
        if (folder/'completion.json').is_file():
            row.update({'status':'complete','selected_epoch':read_json(folder/'completion.json')['best_epoch']})
            for split in ('val3k','lcc_training','lcc_development','lcc_evaluation','lcc_combined'):
                for key,value in read_json(folder/(split+'_metrics.json')).items():row[split+'_'+key]=value
        rows.append(row)
    if not DSF_PREFLIGHT['dsf_enabled']:
        rows.append({'run_id':'P3_DSF_crop15','status':DSF_PREFLIGHT['dsf_status'],'crop_factor':1.5,'geometry_mode':'DYNAMIC not enabled',
          'clean_ce_weight':.75,'spectral_ce_weight':.25})
    c,c_notes=historical_reference('C_crop15',HISTORICAL_C_CROP15_SUMMARY)
    sf,sf_notes=historical_reference('P3_SF_crop15',HISTORICAL_P3_SF_CROP15_SUMMARY);notes=c_notes+sf_notes
    for row in rows:
        for label,ref in (('C',c),('SF',sf)):
            for key in DELTA_COLUMNS:row['delta_vs_'+label+'_'+key]=row[key]-ref[key] if key in row and key in ref else None
        row['source_guard_non_worsened_vs_SF']=(row['val3k_acer']<=sf['val3k_acer']) if 'val3k_acer' in row and 'val3k_acer' in sf else None
    rows.extend([c,sf]);write_json(out/'p3_refinement_comparison.json',rows)
    pd.DataFrame(rows).to_csv(out/'p3_refinement_comparison.csv',index=False)
    contextual=contextual_crop27_rows()
    if contextual:pd.DataFrame(contextual).to_csv(out/'historical_crop27_weight_context.csv',index=False)
    report=['# Bounded P3 refinement: CE-weight × crop1.5 × SF interaction','',
      'Primary base is notebook13 P3_SF_crop15. Notebook13 is preserved. Frozen Train10K/Val3K, DATA_SEED43/TRAIN_SEED100, official PAD-pretrained initialization.',
      'Batch256/AMP/SGD(.005,.9,5e-4), max20/milestones[6,14]/gamma.2, clean warm-up1, earliest-stop8/patience4, Val3K minACER and maxAUC tie-break.',
      'Crop1.5 is matched for Train/Val/LCC. Best is reloaded; only Val3K fits the frozen threshold. LCC is never used for checkpoint selection, early stopping or calibration.',
      'W35=.65 clean+.35 selected-worst CE+aux; W50=.50 clean+.50 selected-worst CE+aux. Aux is unchanged. Warm-up remains clean CE+aux.',
      'Both mandatory runs retain SF centers [0.0833333333,0.25,0.4166666667], sigma0.0555555556 and gain Uniform(.65,.90).',
      'R3_worst_ce35/R4_worst_ce50 existed at crop2.7. This round tests their interaction with the newer crop1.5+SF geometry, not rediscovery of a generic CE-weight optimum.','',
      *metric_table(rows),'','## DSF source-only preflight','',
      'Audited source epoch: '+str(DSF_PREFLIGHT['audited_training_epoch'])+'. Continuous scale comes from the actual crop helper used variable after the exact training jitter.',
      'Boundary interaction/window translation and actual scale reduction are separate; translation alone does not change s_eff.',
      'Decision: '+DSF_PREFLIGHT['dsf_status']+'.',
      'Prespecified gate: deviation >'+str(DSF_SCALE_TOLERANCE)+' in at least '+str(DSF_MIN_VARIATION_FRACTION)+' of samples, and std >'+str(DSF_MIN_SCALE_STD)+'. No target information.','',
      '| Statistic | Training jitter | Unjittered |','|---|---:|---:|']
    for key in ('count','mean','std','min','p05','median','p95','max','fraction_deviation_gt_tolerance','fraction_below_149','fraction_below_145',
      'boundary_interaction_fraction','window_translation_fraction','actual_scale_reduction_fraction'):
        report.append('| '+key+' | '+str(DSF_PREFLIGHT['jittered'][key])+' | '+str(DSF_PREFLIGHT['unjittered'][key])+' |')
    report+=['','Per-sample CSV and summary JSON are included in artifacts. Audit uses source image headers without decoding, with EXIF-aware dimensions.',
      'If enabled, DSF centers_i=[.15,.45,.75]*(s_eff_i/2.7), sigma_i=.10*(s_eff_i/2.7), weights=.75/.25; only preprocessing geometry enters this path.',
      'Static W35/W50 keep the original static generator/selection path. No substitute third experiment or combination is added.','',
      '## Deltas versus C_crop15 and P3_SF_crop15','',
      '| Run | Reference | '+' | '.join(DELTA_COLUMNS)+' |','|---|---|'+'---:|'*len(DELTA_COLUMNS)]
    completed=[r for r in rows if r['status']=='complete']
    for row in rows:
        if row['run_id'] in ('C_crop15','P3_SF_crop15'):continue
        for label in ('C','SF'):
            report.append('| '+row['run_id']+' | '+label+' | '+' | '.join(f'{row["delta_vs_"+label+"_"+key]:+.6f}' if row.get('delta_vs_'+label+'_'+key) is not None else 'pending' for key in DELTA_COLUMNS)+' |')
    report+=['','C pooled AUC .830642 and SF pooled AUC .837525 were supplied by the user. Without attached summaries these are AUC-only references, not newly verified measurements.',
      'Descriptive screening target '+str(DESCRIPTIVE_SCREENING_TARGET)+' is contextual only; no search continues until it is reached.','',
      '## Relevant diagnostics','']
    for row in completed:
        frame=pd.read_csv(out/'runs'/row['run_id']/'spectral_diagnostics.csv');active=frame.loc[frame.phase=='active']
        if not active.empty:
            last=active.iloc[-1]
            report.append('- '+row['run_id']+' last active epoch: '+', '.join(k+f'={last[k]:.6f}' for k in ('clean_ce','worst_ce','train_total_loss','low_fraction','mid_fraction','high_fraction','margin_drop','spectral_flip_rate','sampled_gain_mean'))+'.')
        da=row['delta_vs_SF_lcc_combined_auc'];dh=row.get('delta_vs_SF_lcc_combined_hter')
        if da is not None and da>0:report.append('  Pooled AUC improves descriptively versus SF; inspect source guard and official LCC evaluation for trade-offs.')
        elif dh is not None and dh<0:report.append('  HTER improves without pooled AUC improvement; this may be an operating-point/calibration effect.')
        for ref,label in ((c,'C'),(sf,'SF')):
            for split in ('lcc_evaluation','lcc_combined'):
                for suffix in ('candidate_n','scored_n'):
                    key=split+'_'+suffix
                    if key in ref and row[key]!=ref[key]:notes.append(row['run_id']+' '+key+' differs from historical '+label+'; interpret comparability cautiously.')
    report+=['','## Historical crop2.7 weight context','',
      'These prior measurements contextualize generic CE-weight behavior at crop2.7. New-run deltas measure CE-weight×crop1.5×SF interaction and are not an isolated estimate of crop or geometry effects.']
    report+=metric_table(contextual) if contextual else ['Optional notebook06 measured summaries unavailable; no contextual metrics are invented.']
    report+=['','## Implementation checks','',
      'implementation_sanity_checks.json records .75/.25 scalar/backward equivalence, changed W35/W50 objectives, unchanged clean warm-up, DSF-at1.5 center/sigma/static-mask equivalence and batch/band alignment.',
      'Run configs enforce distinct coefficients and geometry mode. Resume restores committed best/model/optimizer/scheduler/scaler/history/patience/RNG within each run only.',
      'Each completed run receives standalone artifacts/ZIP; overall artifacts are refreshed immediately. No CelebA full Test, detector or LCC fitting.','',
      '## Candidate for full-scale validation','']
    eligible=[r for r in completed if r['lcc_combined_auc']>sf['lcc_combined_auc'] and r['source_guard_non_worsened_vs_SF'] is not False]
    if eligible:
        candidate=max(eligible,key=lambda r:(r['lcc_combined_auc'],-r['val3k_acer']))
        caveat=' Source guard remains pending an attached SF Val3K artifact.' if candidate['source_guard_non_worsened_vs_SF'] is None else ' Source Val3K ACER is not worse than attached SF reference.'
        report.append('At most one descriptive candidate: '+candidate['run_id']+' for full-scale validation, pooled AUC '+f'{candidate["lcc_combined_auc"]:.6f}'+'.'+caveat+' This is not a final method winner or an automatic launch.')
    else:report.append('No completed new variant meets this descriptive SF-improvement/source-guard screen yet. Do not add experiments to chase the target.')
    lookup={r['run_id']:r for r in completed}
    pair=[lookup.get(name) for name in ('P3_DSF_crop15','P3_SF_W35_crop15')]
    if all(pair) and all(r['lcc_combined_auc']>sf['lcc_combined_auc'] and r['source_guard_non_worsened_vs_SF'] is True for r in pair):
        report.append('Both independent DSF and W35 show descriptive evidence with source guards; a future P3_DSF_W35 may be considered. It is not implemented or run here.')
    else:report.append('No geometry+weight combination is recommended now; independent evidence for both DSF and W35 with source guards is required.')
    report+=['','## Limitations','',
      '- Train10K/Val3K screening scale and one training seed.',
      '- LCC has been repeatedly used as an external development benchmark; candidate screening does not establish independent final-test generalization.',
      '- No significance claim; multi-seed robustness is deferred because of compute constraints.',
      '- Source-only preflight uses epoch2 jitter; later epoch draws can differ. Header-derived dimensions track OpenCV orientation, and s_eff is continuous before pixel rounding.',
      '- Attached historical artifacts are needed for full source/LCC deltas and coverage comparability; prompt AUC-only references remain explicitly labeled.',
      '',*notes]
    (out/'p3_refinement_report.md').write_text('\n'.join(report)+'\n')
    return rows

def zip_outputs(name=None):
    if name is not None:
        folder=out/'runs'/name;run_zip=out.parent/(name+'.zip')
        with zipfile.ZipFile(run_zip,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
            for file in sorted(folder.rglob('*')):
                if file.is_file():archive.write(file,(Path('runs')/name/file.relative_to(folder)).as_posix())
            for filename in ('P3_REFINEMENT_CROP15_PROTOCOL_v1.md','dsf_preflight_summary.json','dsf_preflight_samples.csv','implementation_sanity_checks.json'):
                archive.write(out/filename,filename)
        display(FileLink(str(run_zip)))
    root_zip=out.parent/(out.name+'.zip')
    print('[Export] Writing:',root_zip,flush=True)
    with zipfile.ZipFile(root_zip,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for file in sorted(out.rglob('*')):
            if file.is_file():archive.write(file,(Path(out.name)/file.relative_to(out)).as_posix())
    display(FileLink(str(root_zip)))

# Export preflight/report even before training; inspect this cell without executing the final runner.
export_report();zip_outputs()


## Execute selected predefined runs

This cell starts expensive GPU training on Kaggle. Completed run artifacts are exported immediately.

In [ ]:
selected=select_runs(RUN_FILTER)
print('[Runner] Selected predefined runs:',selected,flush=True)
for name in selected:
    try:run_one(name,RUNS[name])
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('Frozen batch256 ran out of GPU memory; stop instead of changing the recipe') from exc
    export_report();zip_outputs(name)
print('Finished selected P3 refinement runs:',selected,flush=True)
